# Adaptive Step Size and Online Calibration

Two halves:

- **Part 1** uses the logistic ODE -- a smooth, easy problem -- to demonstrate **post-hoc calibration** and the relationship between the per-step quasi-MLE and the global MLE.
- **Part 2** uses a **two-component logistic with staggered transitions** -- a smooth problem with two well-separated fast regions -- to demonstrate **adaptive step control**. The per-step quasi-MLE feeds both the local error estimate (driving accept/reject) and the running covariance rescaling.


In [ ]:
import jax
import jax.numpy as np
import jax.scipy.stats as jss
import matplotlib.pyplot as plt
import numpy as onp

from ode_filters.calibration import (
    posthoc_mle_sigma_sqr,
    quasi_mle_sigma_sqr,
    rescale_sqr_seq,
)
from ode_filters.filters import ekf1_sqr_adaptive_loop, ekf1_sqr_loop
from ode_filters.measurement import ODEInformation
from ode_filters.priors import IWP, taylor_mode_initialization

jax.config.update("jax_enable_x64", True)


## Part 1 -- Post-hoc calibration on the logistic ODE

$$\dot x = x(1-x), \qquad x(0)=0.1, \qquad t \in [0, 8].$$

Run a fixed-step EKF with `sigma = 1` (uncalibrated), then calibrate post-hoc.

### What "post-hoc MLE" optimises

Both calibrators come from the *same* Gaussian likelihood. At step $n$ the predicted-observation marginal is $(m_z^{(n)}, S_n)$; rescaling the prior diffusion by $\sigma^2$ turns $S_n$ into $\sigma^2 S_n$. The joint log-likelihood across $N$ steps is

$$\log p(m_z^{(1{:}N)}\mid \sigma^2) = -\tfrac{1}{2}\sum_n \Big[ d\log(2\pi) + \log\!\big|\sigma^2 S_n\big| + \tfrac{1}{\sigma^2}\, m_z^{(n)\top} S_n^{-1} m_z^{(n)}\Big].$$

- Maximising **with one $\sigma_n^2$ per step** gives the per-step quasi-MLE $\widehat\sigma_n^2 = \tfrac{1}{d} m_z^{(n)\top} S_n^{-1} m_z^{(n)}$.
- Maximising **with one global $\sigma^2$ under the constant-sigma assumption** gives the post-hoc MLE $\widehat\sigma^2 = \tfrac{1}{Nd}\sum_n m_z^{(n)\top} S_n^{-1} m_z^{(n)}$.

Same likelihood, one-parameter constraint vs N-parameter. The closed forms give us $\widehat\sigma^2_{\text{post-hoc}} = \tfrac{1}{N}\sum_n \widehat\sigma^2_n$ -- the global MLE is the arithmetic mean of the per-step quasi-MLEs. We verify this numerically below.

In [ ]:
def vf_log(x, *, t):
    return x * (1 - x)


x0_log = np.array([0.1])
tspan_log = (0.0, 8.0)
prior_log = IWP(q=2, d=1)
mu0_log, S0_log = taylor_mode_initialization(vf_log, x0_log, q=2)
measure_log = ODEInformation(vf_log, prior_log.E0, prior_log.E1)

N_log = 10
res_log = ekf1_sqr_loop(mu0_log, S0_log, prior_log, measure_log, tspan_log, N_log)
m_log = np.stack(list(res_log[0]))
P_sqr_log = np.stack(list(res_log[1]))
mz_log = np.stack(list(res_log[-3]))
Pz_sqr_log = np.stack(list(res_log[-2]))
ts_log = onp.linspace(tspan_log[0], tspan_log[1], N_log + 1)

# Per-step quasi-MLE (one number per step).
sigma_sqr_per_step = onp.asarray(
    [float(quasi_mle_sigma_sqr(mz_log[i], Pz_sqr_log[i])) for i in range(N_log)]
)
# Post-hoc joint MLE.
sigma_sqr_post = float(posthoc_mle_sigma_sqr(mz_log, Pz_sqr_log))

print(f"per-step quasi-MLE: mean = {sigma_sqr_per_step.mean():.6g}")
print(f"post-hoc joint MLE:        {sigma_sqr_post:.6g}")
print(f"absolute difference:       {abs(sigma_sqr_per_step.mean() - sigma_sqr_post):.2e}")


The two numbers agree to machine precision. Now rescale the stored covariances by $\sqrt{\widehat\sigma^2}$ and compare to the uncalibrated band.

In [ ]:
P_sqr_log_cal = rescale_sqr_seq(P_sqr_log, sigma_sqr_post)
P_log = np.einsum("nij,nik->njk", P_sqr_log, P_sqr_log)
P_log_cal = np.einsum("nij,nik->njk", P_sqr_log_cal, P_sqr_log_cal)
var_uncal = onp.asarray(P_log[:, 0, 0])
var_cal = onp.asarray(P_log_cal[:, 0, 0])
x_log = onp.asarray(m_log[:, 0])

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(ts_log, x_log, "k-", lw=1.0, label="filtered mean")
ax.fill_between(
    ts_log,
    x_log - 2 * onp.sqrt(var_uncal),
    x_log + 2 * onp.sqrt(var_uncal),
    color="C0", alpha=0.25, label="uncalibrated 2-sigma",
)
ax.fill_between(
    ts_log,
    x_log - 2 * onp.sqrt(var_cal),
    x_log + 2 * onp.sqrt(var_cal),
    color="C1", alpha=0.45, label="post-hoc MLE 2-sigma",
)
ax.set_xlabel("t")
ax.set_ylabel("x(t)")
ax.set_title("logistic: uncalibrated vs post-hoc calibrated uncertainty")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


Calibration shrinks the band by the factor $\sqrt{\widehat\sigma^2}$ (here $\widehat\sigma^2 \ll 1$ because the IWP$(2)$ prior over-quotes uncertainty for this smooth problem). The shape of the band is unchanged -- post-hoc calibration is one scalar applied to every covariance.

## Part 2 -- Adaptive step control on a two-scale problem

The post-hoc MLE assumes one constant $\sigma$ explains the whole trajectory. To stress that assumption we need a problem with *time-varying* scale -- distinct fast and slow regions. A clean choice with an analytic solution is a pair of independent logistic equations with **staggered transitions**:

$$
\dot x_1 = r\, x_1(1 - x_1),  \qquad x_1(0) = 10^{-5},
$$
$$
\dot x_2 = r\, x_2(1 - x_2),  \qquad x_2(0) = 10^{-10},
$$

with $r = 2$ for both. Both components sit near zero for a long while, then transition sharply through $0.5$ on their way to 1. The much smaller $x_2(0)$ delays its transition by about $\Delta t \approx (\log 10^{10} - \log 10^5)/r \approx 5.8$ relative to $x_1$, so the trajectory has *two well-separated fast regions* with a quiet plateau in between. The optimal $h$ varies by more than an order of magnitude between these regions, and each component has the closed-form solution $x_i(t) = 1 / (1 + (1/x_i(0) - 1)\,e^{-r t})$ giving us an analytic reference for accuracy.

In [ ]:
R_RATE = 2.0
X0_VEC = onp.array([1e-5, 1e-10])


def vf_dl(x, *, t):
    return np.array([R_RATE * x[0] * (1 - x[0]), R_RATE * x[1] * (1 - x[1])])


def analytic(t, x0, r):
    return 1.0 / (1.0 + (1.0 / x0 - 1.0) * onp.exp(-r * t))


tspan_dl = (0.0, 15.0)
q_dl = 3
prior_dl = IWP(q=q_dl, d=2)
mu0_dl, S0_dl = taylor_mode_initialization(vf_dl, np.asarray(X0_VEC), q=q_dl)
measure_dl = ODEInformation(vf_dl, prior_dl.E0, prior_dl.E1)

res_adapt = ekf1_sqr_adaptive_loop(
    mu0_dl, S0_dl, prior_dl, measure_dl, tspan_dl,
    atol=1e-5, rtol=1e-3, h_min=1e-9,
)
n_acc = len(res_adapt.h_seq)
h_lo, h_hi = min(res_adapt.h_seq), max(res_adapt.h_seq)
print(
    f"accepted = {n_acc}, rejected = {res_adapt.n_rejected}, "
    f"h in [{h_lo:.3g}, {h_hi:.3g}] -- dynamic range {h_hi / h_lo:.1f}x"
)


In [ ]:
ts_adapt = onp.asarray(res_adapt.t_seq)
m_adapt = onp.stack([onp.asarray(m) for m in res_adapt.m_seq])
x_adapt = m_adapt @ onp.asarray(prior_dl.E0.T)
P_sqr_adapt = onp.stack([onp.asarray(P) for P in res_adapt.P_seq_sqr])
P_adapt = onp.einsum("nij,nik->njk", P_sqr_adapt, P_sqr_adapt)
cov_x_adapt = onp.einsum(
    "ij,njk,lk->nil", onp.asarray(prior_dl.E0), P_adapt, onp.asarray(prior_dl.E0)
)
var_x1 = cov_x_adapt[:, 0, 0]
var_x2 = cov_x_adapt[:, 1, 1]

ts_dense = onp.linspace(tspan_dl[0], tspan_dl[1], 600)
x_true_0 = analytic(ts_dense, float(X0_VEC[0]), R_RATE)
x_true_1 = analytic(ts_dense, float(X0_VEC[1]), R_RATE)
x_true_at_adapt_0 = analytic(ts_adapt, float(X0_VEC[0]), R_RATE)
x_true_at_adapt_1 = analytic(ts_adapt, float(X0_VEC[1]), R_RATE)
max_err = max(
    float(onp.abs(x_adapt[:, 0] - x_true_at_adapt_0).max()),
    float(onp.abs(x_adapt[:, 1] - x_true_at_adapt_1).max()),
)
print(f"max trajectory error vs analytic: {max_err:.2g}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
for ax, var, comp, var_band, x_true_dense in [
    (ax1, 0, "x_1", var_x1, x_true_0),
    (ax2, 1, "x_2", var_x2, x_true_1),
]:
    ax.plot(
        ts_dense, x_true_dense, "k-", lw=1.2, alpha=0.55,
        label="analytic",
    )
    ax.plot(
        ts_adapt, x_adapt[:, var], "o-", color="C2", ms=3.5, lw=0.6,
        label=f"adaptive ({n_acc} accepted)",
    )
    ax.fill_between(
        ts_adapt,
        x_adapt[:, var] - 2 * onp.sqrt(var_band),
        x_adapt[:, var] + 2 * onp.sqrt(var_band),
        color="C2", alpha=0.3, label="adaptive 2-sigma",
    )
    ax.set_ylabel(f"${comp}(t)$")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.15)
ax2.set_xlabel("t")
ax1.set_title("Staggered double-logistic: adaptive run vs analytic")
ax1.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


Accepted-step markers crowd densely at the two transition regions ($t \approx 5.8$ for $x_1$ and $t \approx 11.5$ for $x_2$). Between transitions, when both components are near 0 or near 1, the controller takes long strides. The adaptive trajectory tracks the analytic solution to a few parts in $10^4$.

### Phase plot

Plotting the trajectory in $(x_1, x_2)$ space reveals the staggering directly: the trajectory hugs the bottom-left corner ($x_1 \approx 0, x_2 \approx 0$) for a long while, then sweeps right along the $x_2 \approx 0$ axis as $x_1$ transitions, parks at $(1, 0)$ while waiting for $x_2$, and finally sweeps up to $(1, 1)$. Marker colour encodes $\log_{10} h$ at each accepted step.

In [ ]:
x_true_phase_0 = analytic(ts_dense, float(X0_VEC[0]), R_RATE)
x_true_phase_1 = analytic(ts_dense, float(X0_VEC[1]), R_RATE)

fig, ax = plt.subplots(figsize=(6.2, 5.6))
ax.plot(
    x_true_phase_0, x_true_phase_1, "k-", lw=1.2, alpha=0.55,
    label="analytic trajectory",
)
ax.plot(
    x_adapt[:, 0], x_adapt[:, 1], "-", color="C2", lw=0.6, alpha=0.4,
)
h_arr = onp.asarray(res_adapt.h_seq)
h_marker = onp.concatenate([[h_arr[0]], h_arr])
log_h = onp.log10(h_marker)
sizes = 12 + 80 * (log_h - log_h.min()) / max(log_h.max() - log_h.min(), 1e-12)
scatter = ax.scatter(
    x_adapt[:, 0], x_adapt[:, 1],
    c=log_h, s=sizes, cmap="viridis",
    edgecolors="k", linewidths=0.4, zorder=3,
    label=f"adaptive accepted ({n_acc})",
)
cb = plt.colorbar(scatter, ax=ax, shrink=0.85)
cb.set_label(r"$\log_{10} h$ at this step")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("Phase plot: trajectory and adaptive step density")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Dark markers (small $h$) cluster precisely where the trajectory turns: at the bottom-right corner $(1, 0)$ during the $x_1$ transition and approaching the top-right corner $(1, 1)$ during the $x_2$ transition. Bright yellow markers (large $h$) sit on the smooth horizontal stretch where only $x_1$ moves and along the long initial wait near the origin.

### Step-size and per-step diffusion traces

Two adaptive-run diagnostics every honest report should include. The step-size dynamic range shows the *quantitative* value of adaptive control; the per-step quasi-MLE $\widehat\sigma_n^2$ tracks the same dynamics from the residual side.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
t_step = ts_adapt[1:]  # h_seq[k] is the step that produced ts_adapt[k+1]
ax1.semilogy(t_step, res_adapt.h_seq, "o-", color="C2", ms=3.5)
ax1.set_xlabel("t")
ax1.set_ylabel("h (log)")
ax1.set_title("adaptive step size")
ax1.grid(True, alpha=0.3)
ax2.semilogy(t_step, res_adapt.sigma_sqr_seq, "o-", color="C3", ms=3.5)
ax2.set_xlabel("t")
ax2.set_ylabel(r"$\widehat{\sigma}^2_n$ (log)")
ax2.set_title("per-step quasi-MLE diffusion")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


$h(t)$ drops by an order of magnitude at the two transitions and recovers in between. $\widehat\sigma_n^2(t)$ spikes at the same times -- the prior covariance alone can't explain the residual at a transition, so the calibrator inflates $\sigma$. Between transitions both quantities settle to their slow-region levels.

### Whitened-residual diagnostic: Q-Q plot vs $\mathcal N(0, 1)$

**Why the pooled per-step whitened residual is *not* Gaussian.** The adaptive loop rescales each step's stored $P_z^{1/2}$ by $\sqrt{\widehat\sigma_n^2}$, so the whitened residual computed from the stored covariance is $v_n^{\text{cal}} = v_n^{\text{raw}} / \sqrt{\widehat\sigma_n^2}$ and **by construction** satisfies $\|v_n^{\text{cal}}\|^2 = d$ exactly *every* step. Pooling these forces every $d$-block of samples onto a sphere of radius $\sqrt d$ -- the distribution is *not* $\mathcal N(0, I_d)$, and a Q-Q plot of these is uninformative.

**The right diagnostic** uses a *single global* $\widehat\sigma$ to whiten -- e.g. the running mean $\widehat\sigma^2_{\text{global}} = \tfrac1N \sum_n \widehat\sigma_n^2$ (the post-hoc MLE; see Part 1). Concretely,

$$v_n^{\text{global}} \;=\; \frac{v_n^{\text{raw}}}{\sqrt{\widehat\sigma^2_{\text{global}}}} \;=\; v_n^{\text{cal}} \cdot \sqrt{\widehat\sigma_n^2 / \widehat\sigma^2_{\text{global}}}.$$

Under correct specification with truly constant $\sigma$, $\widehat\sigma_n^2 / \widehat\sigma^2_{\text{global}}$ fluctuates around 1 across steps and the pooled $v_n^{\text{global}}$ is approximately $\mathcal N(0, 1)$.

In [ ]:
sigma_sqr_n = onp.asarray(res_adapt.sigma_sqr_seq)
sigma_global_sqr = float(sigma_sqr_n.mean())
print(f"sigma_global^2 (= mean of per-step sigma_n^2) = {sigma_global_sqr:.3g}")
print(
    f"sigma_n^2 spans  [{sigma_sqr_n.min():.2g}, {sigma_sqr_n.max():.2g}]  "
    f"({sigma_sqr_n.max() / sigma_sqr_n.min():.0f}x)"
)

v_global_blocks = []
for i, (mz, Pz_sqr) in enumerate(
    zip(res_adapt.mz_seq, res_adapt.Pz_seq_sqr, strict=True)
):
    v_stored = jax.scipy.linalg.solve_triangular(Pz_sqr.T, mz, lower=True)
    scale = onp.sqrt(sigma_sqr_n[i] / sigma_global_sqr)
    v_global_blocks.append(onp.asarray(v_stored) * scale)
z = onp.concatenate(v_global_blocks)
n = z.size
n_out = int((onp.abs(z) > 3.0).sum())
expected_out = 2 * (1 - float(jss.norm.cdf(3.0))) * n

print(
    f"n_samples = {n}, mean = {z.mean():+.3f}, std = {z.std():.3f}  "
    f"(target N(0, 1))"
)
print(f"#|z| > 3 outliers = {n_out}  (expected under N(0,1): {expected_out:.1f})")


In [ ]:
sorted_z = onp.sort(z)
p_k = (onp.arange(1, n + 1) - 0.5) / n
theoretical_q = onp.asarray(jss.norm.ppf(p_k))

fig, ax = plt.subplots(figsize=(5.8, 5.8))
lim = 4.0
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1.0, alpha=0.7, label="$y = x$")
for s in (1.0, 2.0, 3.0):
    for sgn in (+1.0, -1.0):
        ax.axhline(sgn * s, color="k", lw=0.4, alpha=0.18)
        ax.axvline(sgn * s, color="k", lw=0.4, alpha=0.18)
in_box = onp.abs(sorted_z) <= lim
ax.scatter(
    theoretical_q[in_box], sorted_z[in_box],
    s=20, color="C2", edgecolors="k", linewidths=0.3,
    label=f"in-axis  ({int(in_box.sum())} / {n})",
)
if (~in_box).any():
    clipped = onp.clip(sorted_z[~in_box], -lim, lim)
    ax.scatter(
        theoretical_q[~in_box], clipped,
        s=44, marker="X", color="C3", edgecolors="k", linewidths=0.4, zorder=4,
        label=f"|z| > {lim:.0f}  ({int((~in_box).sum())}) [clipped]",
    )
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(r"theoretical quantile  $\Phi^{-1}(p_k)$")
ax.set_ylabel("sorted global-whitened residual")
ax.set_title(
    f"Q-Q plot vs $\\mathcal{{N}}(0, 1)$  (n = {n}, mean = {z.mean():+.2f}, std = {z.std():.2f})"
)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


Reading the Q-Q plot:

- The bulk of the points lies on the diagonal $y = x$ with `std(z)` close to 1 -- the constant-$\sigma$ model fits this two-scale problem reasonably well.
- A few **outliers** at the tails come from the transition steps. The per-step quasi-MLE absorbed those into a much larger $\widehat\sigma_n^2$ (visible as spikes in the previous diagnostic plot); re-whitening with the *global* $\widehat\sigma^2$ makes them stand out.

More general Q-Q failure modes:

- Line **parallel to but shifted** from the diagonal: constant mean bias (typically a one-sided EKF1 linearisation error).
- **Slope $\ne 1$**: globally mis-calibrated.
- **S-curve** through the diagonal: heavier-than-Gaussian tails (Student's-$t$ territory).
- **One-sided curl**: skewness, often a regime transition the prior cannot represent.

See the wiki note *whitened-residual-diagnostics* for the full failure-mode table.

## Where to go next

- [Diffusion Calibration](../calibration.md) -- per-step quasi-MLE, post-hoc MLE, aggregator semantics.
- [Adaptive Step-Size Control](../adaptive-steps.md) -- tolerances, the PI and P controllers, failure modes.
